<a href="https://colab.research.google.com/github/ekt4r/playground-series-s6e3/blob/main/lightgbm_tuned.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
import warnings

warnings.filterwarnings('ignore')

In [3]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')
original = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')

In [4]:
train['Churn'] = train['Churn'].map({'Yes': 1, 'No': 0}).astype(int)
original['Churn'] = original['Churn'].map({'Yes': 1, 'No': 0}).astype(int)

original['TotalCharges'] = pd.to_numeric(original['TotalCharges'], errors='coerce')
original['TotalCharges'].fillna(original['TotalCharges'].median(), inplace=True)
# original['TotalCharges'].fillna(original['MonthlyCharges'] * original['tenure'], inplace=True)

In [5]:
original.drop('customerID', axis=1, inplace=True)
train.drop('id', axis=1, inplace=True)
test.drop('id', axis=1, inplace=True)

In [6]:
cats = [
    'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService',
    'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup',
    'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies',
    'Contract', 'PaperlessBilling', 'PaymentMethod'
]

nums = ['tenure', 'MonthlyCharges', 'TotalCharges']

target = 'Churn'

new_nums = []
nums_as_cat = []

In [7]:
for col in ["Contract", "InternetService", "PaymentMethod"]:

    mean_map = train.groupby(col)["MonthlyCharges"].mean()
    train[f"{col}ChargeMean"] = train[col].map(mean_map)
    test[f"{col}ChargeMean"] = test[col].map(mean_map)

    mean_map = train.groupby(col)["tenure"].mean()
    train[f"{col}TenureMean"] = train[col].map(mean_map)
    test[f"{col}TenureMean"] = test[col].map(mean_map)

In [8]:
for col in nums:
    freq = pd.concat([train[col], original[col], test[col]]).value_counts(normalize=True)
    for df in [train, test, original]:
        df[f'Freq{col}'] = df[col].map(freq).fillna(0).astype('float32')
    new_nums.append(f'Freq{col}')

In [9]:
for col in cats:

    churn_mean = original.groupby(col)[target].mean()
    churn_std = original.groupby(col)[target].std()

    train[f"{col}ChurnMean"] = train[col].map(churn_mean)
    test[f"{col}ChurnMean"] = test[col].map(churn_mean)

    train[f"{col}ChurnStd"] = train[col].map(churn_std)
    test[f"{col}ChurnStd"] = test[col].map(churn_std)

    freq = pd.concat([train[col], test[col]]).value_counts(normalize=True)

    train[f"{col}Freq"] = train[col].map(freq)
    test[f"{col}Freq"] = test[col].map(freq)

In [10]:
q1 = train["MonthlyCharges"].quantile(0.25)
q3 = train["MonthlyCharges"].quantile(0.75)

for df in [train, test]:

    df["ChargeOutlier"] = (
        (df["MonthlyCharges"] < q1) |
        (df["MonthlyCharges"] > q3)
    ).astype(int)

    df["TenureSquared"] = df["tenure"] ** 2
    df["MonthlyChargesSquared"] = df["MonthlyCharges"] ** 2

    df['MCLog'] = np.log1p(df['MonthlyCharges'])
    df['TCLog'] = np.log1p(df['TotalCharges'])

    df['MCSqueezed'] = 1 / (df['MonthlyCharges'] + 1)
    df['TCSqueezed'] = 1 / (df['TotalCharges'] + 1)

    df["TenureLog"] = np.log1p(df["tenure"])
    df['TenureSqueezed'] = 1 / (df['tenure'] + 1)

    df["AvgMonthlySpend"] = df["TotalCharges"] / (df["tenure"] + 1)
    df["SpendDeviation"] = df["MonthlyCharges"] - df["AvgMonthlySpend"]
    df["MonthlyToTotalRatio"] = df["MonthlyCharges"] / (df["TotalCharges"] + 1)

    df["IsMonthToMonth"] = (df["Contract"] == "Month-to-month").astype(int)

    df["IsElectronicCheck"] = (df["PaymentMethod"] == "Electronic check").astype(int)
    df["AutoPayment"] = df["PaymentMethod"].str.contains("automatic").astype(int)

    df["SeniorCitizenMonthlyInteraction"] = (
        df["SeniorCitizen"].astype(int) * df["MonthlyCharges"]
    )

    df["ServiceCount"] = (
        (df["OnlineSecurity"] == "Yes").astype(int) +
        (df["OnlineBackup"] == "Yes").astype(int) +
        (df["DeviceProtection"] == "Yes").astype(int) +
        (df["TechSupport"] == "Yes").astype(int) +
        (df["StreamingTV"] == "Yes").astype(int) +
        (df["StreamingMovies"] == "Yes").astype(int)
    )

    df["IsFiber"] = (df["InternetService"] == "Fiber optic").astype(int)

    df["FiberHighCharge"] = (
        (df["InternetService"] == "Fiber optic") &
        (df["MonthlyCharges"] > df["MonthlyCharges"].median())
    ).astype(int)

    df["FiberNoSupport"] = (
        (df["InternetService"] == "Fiber optic") &
        (df["TechSupport"] == "No")
    ).astype(int)

    df["EarlyTenure"] = (df["tenure"] < 12).astype(int)
    df["HighChargeEarly"] = (
        (df["tenure"] < 12) &
        (df["MonthlyCharges"] > df["MonthlyCharges"].median())
    ).astype(int)

    df["TenureBucket"] = pd.cut(
        df["tenure"],
        bins=[0, 6, 12, 24, 48, 72],
        labels=False
    )

    df["VeryEarly"] = (df["tenure"] < 6).astype(int)

    df["LifetimeValue"] = df["MonthlyCharges"] * df["tenure"]

    df["ChargeRank"] = df["MonthlyCharges"].rank(pct=True)
    df["TenureRank"] = df["tenure"].rank(pct=True)

    df["Contract_Internet"] = (
        df["Contract"].astype(str) + "_" +
        df["InternetService"].astype(str)
    )

    df["Contract_Payment"] = (
        df["Contract"].astype(str) + "_" +
        df["PaymentMethod"].astype(str)
    )

new_nums = [col for col in df.columns.tolist() if col not in (nums + cats + [target])]

In [11]:
for col in cats + nums:
    tmp = original.groupby(col)[target].mean()
    name = f"OrigP{col}"
    train = train.merge(tmp.rename(name), on=col, how="left")
    test = test.merge(tmp.rename(name), on=col, how="left")
    for df in [train, test]:
        df[name] = df[name].fillna(0.5).astype('float32')
    new_nums.append(name)

In [12]:
X, y = train.drop(target, axis=1), train[target]

In [13]:
from sklearn.preprocessing import OneHotEncoder, LabelEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

label_cols = [
    'gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
    'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
    'TechSupport', 'StreamingTV', 'StreamingMovies', 'PaperlessBilling'
]

onehot_cols = ['PaymentMethod', 'Contract_Internet', 'Contract_Payment']

ord_mapping = {
    'Contract': ['Two year', 'One year', 'Month-to-month'],
    'MultipleLines': ['No phone service', 'No', 'Yes'],
    'InternetService': ['No', 'DSL', 'Fiber optic'],
    'OnlineSecurity': ['No internet service', 'No', 'Yes'],
    'OnlineBackup': ['No internet service', 'No', 'Yes'],
    'DeviceProtection': ['No internet service', 'No', 'Yes'],
    'TechSupport': ['No internet service', 'No', 'Yes'],
    'StreamingTV': ['No internet service', 'No', 'Yes'],
    'StreamingMovies': ['No internet service', 'No', 'Yes'],
    'Partner': ['No', 'Yes'],
    'Dependents': ['No', 'Yes'],
    'PhoneService': ['No', 'Yes'],
    'PaperlessBilling': ['No', 'Yes'],
    'gender': ['Male', 'Female']
}

ord_cols = list(ord_mapping.keys())

preprocessor = ColumnTransformer(
    transformers=[
        ('onehot', OneHotEncoder(drop='first', sparse_output=False), onehot_cols),
        ('ordinal', OrdinalEncoder(categories=[ord_mapping[col] for col in ord_cols]), ord_cols)
    ],
    remainder='passthrough'
)

pipeline = Pipeline([
    ('preprocessor', preprocessor)
])

pipeline.set_output(transform="pandas")

X_train_encoded = pipeline.fit_transform(X)
X_test_encoded = pipeline.transform(test)

In [19]:
import optuna
import lightgbm as lgb

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

def objective(trial):

    params = {
        "n_estimators": 5000,
        "objective": "binary",
        "metric": "auc",
        "boosting_type": "gbdt",
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.5, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 16, 256),
        "max_depth": trial.suggest_int("max_depth", 1, 12),
        "min_child_samples": trial.suggest_int("min_child_samples", 10, 200),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 0.0, 10.0),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.0, 10.0),
        "objective": "binary",
        "metric": "auc",
        "device": "cpu",
        "n_jobs": -1,
        "verbosity": -1,
        "random_state": 42
    }

    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

    oof = np.zeros(len(X_train_encoded))

    for train_idx, valid_idx in skf.split(X_train_encoded, y):

        X_train, X_valid = X_train_encoded.iloc[train_idx], X_train_encoded.iloc[valid_idx]
        y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]

        model = lgb.LGBMClassifier(**params)

        model.fit(
            X_train,
            y_train,
            eval_set=[(X_valid, y_valid)],
            eval_metric="auc",
            callbacks=[lgb.early_stopping(200), lgb.log_evaluation(0)]
        )

        preds = model.predict_proba(X_valid)[:, 1]
        oof[valid_idx] = preds

    score = roc_auc_score(y, oof)

    return score

In [21]:
study = optuna.create_study(
    direction="maximize"
)

study.optimize(objective, n_trials=300)

[I 2026-03-10 18:35:58,534] A new study created in memory with name: no-name-9902698f-ecec-4018-bba8-b3525bdf7e54


Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[28]	valid_0's auc: 0.914998
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[31]	valid_0's auc: 0.915776
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[41]	valid_0's auc: 0.914248


[I 2026-03-10 18:36:27,332] Trial 0 finished with value: 0.9149989800756317 and parameters: {'learning_rate': 0.4503704279273723, 'num_leaves': 254, 'max_depth': 7, 'min_child_samples': 182, 'subsample': 0.9148066254681142, 'colsample_bytree': 0.9893814977281222, 'reg_alpha': 1.859667436082254, 'reg_lambda': 0.41471035358424224}. Best is trial 0 with value: 0.9149989800756317.


Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[122]	valid_0's auc: 0.916532
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[120]	valid_0's auc: 0.917217
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[86]	valid_0's auc: 0.915673


[I 2026-03-10 18:37:02,921] Trial 1 finished with value: 0.9164705361357262 and parameters: {'learning_rate': 0.2100763699588271, 'num_leaves': 68, 'max_depth': 8, 'min_child_samples': 38, 'subsample': 0.8894668222562755, 'colsample_bytree': 0.7094192489244319, 'reg_alpha': 7.318303932807707, 'reg_lambda': 5.281245955280694}. Best is trial 1 with value: 0.9164705361357262.


...

[I 2026-03-11 08:23:08,996] Trial 297 finished with value: 0.9170545328470795 and parameters: {'learning_rate': 0.04564454713965065, 'num_leaves': 114, 'max_depth': 4, 'min_child_samples': 132, 'subsample': 0.7861052173103136, 'colsample_bytree': 0.6302115471837223, 'reg_alpha': 5.635257803422287, 'reg_lambda': 0.7858889147529153}. Best is trial 226 with value: 0.917108548835282.


Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[1654]	valid_0's auc: 0.916881
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[1345]	valid_0's auc: 0.917555
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[2058]	valid_0's auc: 0.916236


[I 2026-03-11 08:25:12,981] Trial 298 finished with value: 0.9168830539834842 and parameters: {'learning_rate': 0.03215981119028488, 'num_leaves': 24, 'max_depth': 5, 'min_child_samples': 144, 'subsample': 0.796211443623892, 'colsample_bytree': 0.6201623188394016, 'reg_alpha': 0.5563598832359675, 'reg_lambda': 0.526155159719242}. Best is trial 226 with value: 0.917108548835282.


Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[2670]	valid_0's auc: 0.91689
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[3803]	valid_0's auc: 0.917824
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[3378]	valid_0's auc: 0.916359


[I 2026-03-11 08:28:07,866] Trial 299 finished with value: 0.9170206392834277 and parameters: {'learning_rate': 0.05028142279916843, 'num_leaves': 101, 'max_depth': 3, 'min_child_samples': 155, 'subsample': 0.8137498433985054, 'colsample_bytree': 0.6419732060766619, 'reg_alpha': 6.9523562783814885, 'reg_lambda': 0.19986029733781993}. Best is trial 226 with value: 0.917108548835282.


In [22]:
import json

with open('lightgbm-params-2.json', 'w') as f:
    json.dump(study.best_params, f, indent=4)

In [43]:
import requests
import os
from dotenv import load_dotenv

load_dotenv()

TOKEN = os.getenv('TELEGRAM_TOKEN')
CHAT_ID = os.getenv('TELEGRAM_CHAT_ID')

def send_telegram(msg):
    url = f'https://api.telegram.org/bot{TOKEN}/sendMessage'
    try:
        requests.post(
            url,
            data={
                'chat_id': CHAT_ID,
                'text': msg
            }
        )
    except:
        pass

In [44]:
send_telegram('Hello, Max!')

In [35]:
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
oof_preds = np.zeros(len(X_train_encoded))
fold_test_preds = np.zeros(len(X_test_encoded))

for fold, (train_idx, valid_idx) in enumerate(skf.split(X_train_encoded, y)):

    print(f'Fold {fold+1}')

    X_train, X_valid = X_train_encoded.iloc[train_idx], X_train_encoded.iloc[valid_idx]
    y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]

    model = lgb.LGBMClassifier(
        n_estimators=10000,
        **study.best_params,
        random_state=123
    )

    model.fit(
        X_train,
        y_train,
        eval_set=[(X_valid, y_valid)],
        eval_metric="auc",
        callbacks=[lgb.early_stopping(300), lgb.log_evaluation(0)]
    )

    oof_preds[valid_idx] = model.predict_proba(X_valid)[:, 1] 
    fold_test_preds += model.predict_proba(X_test_encoded)[:, 1] / skf.n_splits

    score = roc_auc_score(y_valid, oof_preds[valid_idx])
    send_telegram(f'Fold {fold+1} AUC: {score:.5f}')

Fold 1
Training until validation scores don't improve for 300 rounds
Early stopping, best iteration is:
[3238]	valid_0's auc: 0.916114	valid_0's binary_logloss: 0.298227
Fold 2
Training until validation scores don't improve for 300 rounds
Early stopping, best iteration is:
[3355]	valid_0's auc: 0.919538	valid_0's binary_logloss: 0.292559
Fold 3
Training until validation scores don't improve for 300 rounds
Early stopping, best iteration is:
[2547]	valid_0's auc: 0.916471	valid_0's binary_logloss: 0.297624
Fold 4
Training until validation scores don't improve for 300 rounds
Early stopping, best iteration is:
[2443]	valid_0's auc: 0.916372	valid_0's binary_logloss: 0.297811
Fold 5
Training until validation scores don't improve for 300 rounds
Early stopping, best iteration is:
[3254]	valid_0's auc: 0.918027	valid_0's binary_logloss: 0.295014


In [36]:
cv_score = roc_auc_score(y, oof_preds)

print("CV AUC:", cv_score)
send_telegram(f"CV AUC: {cv_score:.5f}")

CV AUC: 0.9172994056375059
